# 01 -- Problem & Data

## Predictive Maintenance: Why It Matters

Predictive maintenance is about catching equipment failures *before* they happen. Instead of waiting for a machine to break (reactive) or replacing parts on a fixed schedule (preventive), we use sensor data to predict when a failure is likely. This reduces downtime, saves money, and keeps production running.

In this project, we work with the **AI4I2020 Predictive Maintenance Dataset** -- a synthetic dataset that simulates machine operating conditions and failure events. The goal is to build a model that predicts **machine failure** from operating conditions like temperature, rotational speed, torque, and tool wear.

### What We're Trying to Predict

- **Target**: `Machine failure` (binary: 0 = normal operation, 1 = failure)
- **Inputs**: Machine type, air temperature, process temperature, rotational speed, torque, tool wear
- **Objective**: Early warning -- flag machines at risk so maintenance can be scheduled proactively

### What Success Looks Like

A good predictive maintenance model doesn't just maximize accuracy. Because failures are rare (~3.4% in this dataset), a model that always predicts "normal" would achieve 96.6% accuracy but catch *zero* failures. That's useless.

Instead, we care about:

- **Recall**: Of all actual failures, how many did we catch? (Missing a failure is costly)
- **Precision**: Of all predicted failures, how many were real? (False alarms waste time)
- **F1-score**: Balance between recall and precision
- **PR-AUC**: Area under the Precision-Recall curve -- robust for imbalanced data

We'll prioritize **recall** because missing a real failure is more costly than investigating a false alarm.

In [1]:
# Setup
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path

from src.data import load_data, get_feature_target, get_feature_info

## Load the Dataset

In [2]:
df = load_data()
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset shape: (10000, 14)
Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [4]:
df.describe()

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2886.89568,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,1.00000,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,2500.75000,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,5000.50000,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,7500.25000,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,10000.00000,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


## Understand the Target: Machine Failure

In [5]:
target = df["Machine failure"]
failures = target.sum()
non_failures = len(target) - failures
fail_pct = failures / len(target) * 100

print(f"Total records: {len(target):,}")
print(f"Failures (1): {failures:,}")
print(f"Non-failures (0): {non_failures:,}")
print(f"Failure rate: {fail_pct:.2f}%")
print(f"\nTarget distribution:")
print(target.value_counts().sort_index())

Total records: 10,000
Failures (1): 339
Non-failures (0): 9,661
Failure rate: 3.39%

Target distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


### Class Imbalance

The target is **highly imbalanced**: only ~3.4% of records are failures. This has major implications:

- **Accuracy is misleading**: A dummy model predicting "always normal" gets 96.6% accuracy but catches 0% of failures.
- **We need metrics that handle imbalance**: Recall, F1, PR-AUC.
- **We need stratified splits**: Train/test splits must preserve the failure rate.
- **Class weights help**: Most models support `class_weight="balanced"` to penalize missing the minority class more heavily.

This imbalance is *typical* for predictive maintenance -- failures are rare events by definition.

## Identify Features: What's Available?

In [6]:
info = get_feature_info()
for k, v in info.items():
    print(f"{k}: {v}")

numeric_features: ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
categorical_features: ['Type']
target: Machine failure
excluded_failure_modes: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
excluded_identifiers: ['UDI', 'Product ID']


### Feature Categories

| Category | Columns |
|----------|---------|
| **Target** | `Machine failure` |
| **Input features (numeric)** | Air temperature, Process temperature, Rotational speed, Torque, Tool wear |
| **Input features (categorical)** | Type (L, M, H) |
| **Identifiers (no predictive value)** | UDI, Product ID |
| **Failure modes (leakage risk)** | TWF, HDF, PWF, OSF, RNF |

### The Leakage Question: Failure Mode Columns

The columns `TWF`, `HDF`, `PWF`, `OSF`, `RNF` indicate *specific failure types*:

- **TWF**: Tool Wear Failure
- **HDF**: Heat Dissipation Failure
- **PWF**: Power Failure
- **OSF**: Overstrain Failure
- **RNF**: Random Failure

These are **problematic for an early-warning model**. They represent information that is only known *at or after* the moment of failure -- a diagnosis, not a predictor. If we include them, the model "cheats" by using the failure diagnosis to predict the failure.

**Decision**: Exclude failure mode columns from features. We keep only operating conditions that would be available *before* a failure occurs.

## Data Quality Check

In [7]:
print("Missing values:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

print("\nValue ranges for numeric features:")
numeric_cols = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
for col in numeric_cols:
    print(f"  {col}: {df[col].min():.1f} - {df[col].max():.1f}")

print(f"\nMachine type distribution:")
print(df["Type"].value_counts().sort_index())

Missing values:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Duplicate rows: 0

Value ranges for numeric features:
  Air temperature [K]: 295.3 - 304.5
  Process temperature [K]: 305.7 - 313.8
  Rotational speed [rpm]: 1168.0 - 2886.0
  Torque [Nm]: 3.8 - 76.6
  Tool wear [min]: 0.0 - 253.0

Machine type distribution:
Type
H    1003
L    6000
M    2997
Name: count, dtype: int64


### Data Quality Summary

- **No missing values** -- clean dataset
- **No duplicates** -- each record is unique
- **Reasonable ranges** -- all numeric features within expected bounds
- **Balanced machine types** -- L (60%), M (30%), H (10%)

The dataset is clean and ready for exploration.

## What We Learned

1. **Problem**: Predict machine failure from operating conditions (early warning).
2. **Dataset**: 10,000 records, 14 columns, synthetic but realistic.
3. **Target**: Binary, highly imbalanced (3.4% failure rate).
4. **Features**: 6 predictive features (5 numeric + 1 categorical), plus identifiers and failure-mode columns.
5. **Key decision**: Exclude failure-mode columns (TWF, HDF, PWF, OSF, RNF) to avoid data leakage -- they represent post-failure diagnoses.
6. **Data quality**: Clean, no missing values, no duplicates.

---

**Next**: In Notebook 02, we'll explore how operating conditions relate to failures -- visualizing distributions, comparing failed vs. healthy machines, and looking for patterns that inform modeling.